# VRT Super Resolution for Disaster Imagery
This notebook consolidates the implementation into a single Kaggle/Colab-ready pipeline. It clones the official VRT repository, artificially downscales a disaster image, processes it through the Video Restoration Transformer (VRT) using memory-efficient tiling to avoid GPU OOM, and compares the upscaled result against the original input.

## Step 1: Setup Environment
Clone the VRT repository and install the required dependencies (timm, basicsr, etc.).

In [ ]:
import os

# Clone VRT repository if it doesn't exist (silently)
if not os.path.exists('VRT'):
    !git clone -q https://github.com/JingyunLiang/VRT.git

# Change directory into VRT (silently)
%cd -q VRT

# Install required dependencies (silently)
!pip install -q gdown timm torchvision matplotlib opencv-python
!pip install -q basicsr

print('✅ Environment setup and installation complete!')

## Step 2: Prepare the Image and Create Synthetic Low-Res Data
VRT expects a video (sequence of frames). We will take a single high-resolution image, downscale it by 4x to simulate poor quality, and duplicate it 4 times to create a fake video sequence.


In [ ]:
import cv2
import os

# 1. Define paths
high_res_source = '/kaggle/input/datasets/abrar2222864642/super-resolution-dataset/Input/disaster_image.jpg'

if not os.path.exists(high_res_source):
    # Fallback to create a dummy image if user forgot to upload one
    print("Warning: disaster_image.jpg not found. Creating a dummy image for testing...")
    import numpy as np
    os.makedirs('./sample_data/', exist_ok=True)
    high_res_source = './sample_data/dummy_image.jpg'
    dummy_img = np.random.randint(0, 255, (400, 400, 3), dtype=np.uint8)
    cv2.imwrite(high_res_source, dummy_img)

# 2. Load the image
src_img = cv2.imread(high_res_source)
h, w, _ = src_img.shape
print(f"✅ Found source image at: {high_res_source}")
print(f"📐 Original High-Res Size: {w}x{h}")

# 3. Mathematically downscale it by 4x using a clean bicubic filter
target_w, target_h = w // 4, h // 4
low_res_img = cv2.resize(src_img, (target_w, target_h), interpolation=cv2.INTER_CUBIC)
print(f"📉 Synthetically Downscaled Size: {target_w}x{target_h}")

# 4. Create the VRT input folder structure
vrt_input_dir = './testsets/custom_disaster/000'
os.makedirs(vrt_input_dir, exist_ok=True)

# 5. Save it 4 times sequentially to simulate the VRT pseudo-video stream
for frame_idx in range(4):
    frame_name = f"{frame_idx:04d}.png"
    cv2.imwrite(os.path.join(vrt_input_dir, frame_name), low_res_img)

print("🎉 Success! Your synthetic low-resolution sequence is staged and ready for testing.")

## Step 3: Run VRT Inference
We will now run the VRT evaluation script.
- `--task 001_VRT_videosr_bi_REDS_6frames` selects the pre-trained model.
- `--tile 40 128 128` prevents Out-of-Memory (OOM) errors on 16GB GPUs like the Kaggle T4 by splitting the image into processing chunks.

In [ ]:
!python main_test_vrt.py \
  --task 001_VRT_videosr_bi_REDS_6frames \
  --folder_lq testsets/custom_disaster \
  --tile 40 128 128 \
  --tile_overlap 2 20 20 \
  --num_workers 2 \
  --save_result

## Step 4: Evaluate and Visualize the Results
We will extract a 100x100 pixel patch from the center of both the degraded image and the VRT upscaled output, and plot them side-by-side to observe the micro-structural reconstruction.

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# 1. Establish pathways (Relative to VRT folder)
original_path = './testsets/custom_disaster/000/0000.png' # We use the downscaled one to see what VRT saw
upscaled_path = './results/001_VRT_videosr_bi_REDS_6frames/custom_disaster/000/0000.png' 

if not os.path.exists(upscaled_path):
    # Fallback if path structure differs slightly
    upscaled_path = './results/001_VRT_videosr_bi_REDS_6frames/000/0000.png' 

# Load the images safely
original_img = cv2.imread(original_path)
upscaled_img = cv2.imread(upscaled_path)

if original_img is None or upscaled_img is None:
    print("Error: Could not load the images. Ensure the inference step completed successfully.")
else:
    # Convert from BGR to RGB for accurate plotting
    original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    upscaled_img = cv2.cvtColor(upscaled_img, cv2.COLOR_BGR2RGB)

    # 2. Dynamically calculate the image dimensions
    h_org, w_org, _ = original_img.shape
    h_up, w_up, _ = upscaled_img.shape

    print(f"📐 Input Dimensions: {w_org}x{h_org}")
    print(f"📐 Upscaled Dimensions: {w_up}x{h_up}")

    # 3. 100px patch directly from the absolute center
    crop_size_org = min(100, w_org, h_org)
    x_org = w_org // 2 - (crop_size_org // 2)
    y_org = h_org // 2 - (crop_size_org // 2)
    patch_org = original_img[y_org:y_org+crop_size_org, x_org:x_org+crop_size_org]

    # Scale coordinates dynamically by the exact upscale factor for the high-res image
    scale_factor_x = w_up // w_org
    scale_factor_y = h_up // h_org

    crop_size_up = crop_size_org * scale_factor_x
    x_up = x_org * scale_factor_x
    y_up = y_org * scale_factor_y
    patch_up = upscaled_img[y_up:y_up+crop_size_up, x_up:x_up+crop_size_up]

    # 4. Plot side-by-side to expose the micro-structural differences
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    axes[0].imshow(patch_org)
    axes[0].set_title(f"Degraded Input Center Crop ({w_org}x{h_org})\n[Notice Pixel Blocks & Blurring]", fontsize=11)
    axes[0].axis('off')

    axes[1].imshow(patch_up)
    axes[1].set_title(f"VRT Super-Resolved Center Crop ({w_up}x{h_up})\n[Reconstructed Sharp Textures]", fontsize=11)
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()
